# 02 - Univariate GARCH models

This notebook fits the univariate GARCH-family models required before the DCC step.

It uses the existing project package:

- data/preprocessing from `garch_btc_sp.data`,
- statistical/model code from `garch_btc_sp.models`,
- assets already used in the repository: `BTC`, `SP500`, `VIX`, `OIL`, `GOLD`.

No generated CSV/PNG files are committed by this notebook.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"Project root: {ROOT}")

In [ ]:
import pandas as pd
from IPython.display import display

from garch_btc_sp.data.preprocessing import build_yahoo_returns
from garch_btc_sp.models import GARCH_VARIANTS, compare_models, fit_model_grid, select_best_by_bic

print("Available model variants:", GARCH_VARIANTS)

## 1. Load returns from the existing data pipeline

The notebook first tries to read `data/processed/returns_yahoo.parquet`, which is already produced by the repository data step. If it is missing, it falls back to `build_yahoo_returns()` from `garch_btc_sp.data.preprocessing`.

In [ ]:
returns_path = ROOT / "data" / "processed" / "returns_yahoo.parquet"

if returns_path.exists():
    returns = pd.read_parquet(returns_path)
else:
    returns = build_yahoo_returns()

returns = returns.dropna()
print(returns.shape)
display(returns.head())

## 2. Fit 12 model combinations per asset

For each asset, this fits:

- GARCH(1,1),
- EGARCH(1,1),
- GJR-GARCH/TGARCH(1,1),
- APARCH(1,1),

each with `normal`, `t`, and `skewt` innovations.

In [ ]:
all_comparisons = []
best_residuals = {}
best_volatility = {}

for asset in returns.columns:
    print(f"Fitting models for {asset}...")
    fitted = fit_model_grid(returns[asset])
    comparison = compare_models(fitted)
    all_comparisons.append(comparison)

    best_row = select_best_by_bic(comparison).iloc[0]
    best_model = next(
        model
        for model in fitted
        if model.variant == best_row["model"] and model.distribution == best_row["distribution"]
    )
    best_residuals[asset] = best_model.standardized_residuals.rename(asset)
    best_volatility[asset] = best_model.conditional_volatility.rename(asset)

comparison_all = pd.concat(all_comparisons, ignore_index=True)
best_models = select_best_by_bic(comparison_all)
display(best_models)

## 3. Model comparison table

Lower BIC is used for model selection. Ljung-Box p-values are reported for standardized residual diagnostics.

In [ ]:
display(comparison_all.sort_values(["asset", "bic"]))

## 4. Standardized residuals for DCC

These residuals are the output needed by the DCC-GARCH stage.

In [ ]:
standardized_residuals = pd.concat(best_residuals.values(), axis=1).dropna()
conditional_volatility = pd.concat(best_volatility.values(), axis=1).dropna()

print("Standardized residuals:", standardized_residuals.shape)
display(standardized_residuals.head())

print("Conditional volatility:", conditional_volatility.shape)
display(conditional_volatility.head())